<h2>Covid19_Project</h2>

### Problem Statement:
Given data about COVID-19 patients, write code to visualize the impact and
analyze the trend of rate of infection and recovery as well as make predictions
about the number of cases expected a week in future based on the current
trends.  
Dataset:  
CSV and Excel files containing data about the number of COVID-19 confirmed
deaths and recovered patients both around the world and in India. Download Link
Guidelines:  
-Usepandas to accumulate data from multiple data files.  
-Useplotly (visualization library) to create interactive visualizations.  
-UseFacebook prophet library to make time series models.  
-Visualize the prediction by combining these technologies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet

In [ ]:
covid19 = pd.read_csv("C:/Users/AHSAN RAZA/Desktop/covid_19_clean_complete.csv")

In [ ]:
print("\nThe dataset is: \n",covid19)

In [ ]:
# Exploratory Data Analysis

In [ ]:
# Understanding the Data

In [ ]:
print(covid19.head(10))
print(covid19.tail(10))

In [ ]:
covid19.shape

In [ ]:
covid19.columns

In [ ]:
covid19.dtypes

In [ ]:
covid19['Date'] = pd.to_datetime(covid19['Date'], format = '%Y-%m-%d')

In [ ]:
covid19.info()

In [ ]:
covid19.describe()

In [ ]:
object_cols = covid19.select_dtypes(include=['object'])
for col in object_cols.columns:
    print(f"Value counts for column: {col}")
    print(object_cols[col].value_counts())
    print("\n"+"-"*40+"\n")

In [ ]:
# Handle missing values for columns

In [ ]:
covid19.isnull().sum()

In [ ]:
thresholds_column = {
    "numeric": 50,
    "datetime": 30,
    "object": 60
}

numeric_column = covid19.select_dtypes(include=["number"])
datetime_column = covid19.select_dtypes(include=["datetime"])
object_column = covid19.select_dtypes(include=["object"])

print("Numeric columns: \n", numeric_column.columns.tolist())
print("DateTime columns: \n", datetime_column.columns.tolist())
print("Object columns: \n", object_column.columns.tolist())

missing_numeric = numeric_column.isnull().mean()*100
missing_datetime = datetime_column.isnull().mean()*100
missing_object = object_column.isnull().mean()*100

In [ ]:
# For numeric columns
numeric_keep_mask = missing_numeric < thresholds_column["numeric"]
print("Keeping numeric columns: ",numeric_keep_mask[numeric_keep_mask].index.tolist())
numeric_column = numeric_column.loc[:, numeric_keep_mask]

In [ ]:
# For datetime columns
datetime_keep_mask = missing_datetime < thresholds_column["datetime"]
print("Keeping datetime column: ",datetime_keep_mask[datetime_keep_mask].index.tolist())
datetime_column = datetime_column.loc[:, datetime_keep_mask]

In [ ]:
# For object columns
object_keep_mask = missing_object < thresholds_column["object"]
print("Keeping object column: ",object_keep_mask[object_keep_mask].index.tolist())
object_column = object_column.loc[:, object_keep_mask]

In [ ]:
covid19 = pd.concat([numeric_column, datetime_column, object_column], axis=1)

print("\n Cleaned Data Preview:")
print(covid19.info())
print(covid19.head())

In [ ]:
covid19.isnull().sum()

In [ ]:
# Handling missing values for Rows

In [ ]:
row_thresholds = 0.3

max_missing_allowed = int(row_thresholds * covid19.shape[1])

covid19 = covid19[covid19.isna().sum(axis=1) <= max_missing_allowed]

In [ ]:
if covid19.isna().values.any():
    print("Some rows still contain missing values.")
else:
    print("No missing values are present in the rows.")

In [ ]:
# Group data by Date and sum values
# This gives total global numbers per day
daily_data = covid19.groupby("Date")[["Confirmed", "Recovered", "Deaths"]].sum().reset_index()

# Calculate Active cases
daily_data["Active"] = (
    daily_data["Confirmed"] - daily_data["Recovered"] - daily_data["Deaths"]
)

In [ ]:
# Univariate Analysis

In [ ]:
# For Numeric columns

In [ ]:
px.line(
    daily_data,
    x="Date",
    y="Confirmed",
    title="Univariate: Confirmed Cases Over Time"
).show()

px.line(
    daily_data,
    x="Date",
    y="Deaths",
    title="Univariate: Deaths Over Time"
).show()

px.line(
    daily_data,
    x="Date",
    y="Recovered",
    title="Univariate: Recovered Cases Over Time"
).show()

px.line(
    daily_data,
    x="Date",
    y="Active",
    title="Univariate: Active Cases Over Time"
).show()

In [ ]:
px.histogram(
    daily_data,
    x="Confirmed",
    title="Univariate: Distribution of Confirmed Cases"
).show()

px.histogram(
    daily_data,
    x="Deaths",
    title="Univariate: Distribution of Deaths"
).show()

In [ ]:
# For Categorical columns

In [ ]:
px.bar(
    covid19["Country/Region"].value_counts().head(10),
    title="Univariate: Top 10 Countries by Records"
).show()

px.bar(
    covid19["WHO Region"].value_counts(),
    title="Univariate: WHO Region Distribution"
).show()

In [ ]:
# Bivariate Analysis

In [ ]:
# Date vs Numeric

In [ ]:
px.line(
    daily_data,
    x="Date",
    y="Confirmed",
    title="Bivariate: Date vs Confirmed"
).show()

px.line(
    daily_data,
    x="Date",
    y="Deaths",
    title="Bivariate: Date vs Deaths"
).show()

px.line(
    daily_data,
    x="Date",
    y="Recovered",
    title="Bivariate: Date vs Recovered"
).show()

px.line(
    daily_data,
    x="Date",
    y="Active",
    title="Bivariate: Date vs Active"
).show()

In [ ]:
# Numeric vs Numeric

In [ ]:
px.scatter(
    daily_data,
    x="Confirmed",
    y="Recovered",
    title="Bivariate: Confirmed vs Recovered"
).show()

px.scatter(
    daily_data,
    x="Confirmed",
    y="Deaths",
    title="Bivariate: Confirmed vs Deaths"
).show()

px.scatter(
    daily_data,
    x="Confirmed",
    y="Active",
    title="Bivariate: Confirmed vs Active"
).show()

In [ ]:
# Categorical vs Numeric

In [ ]:
px.box(
    covid19,
    x="WHO Region",
    y="Confirmed",
    title="Bivariate: WHO Region vs Confirmed"
).show()

top_countries = (
    covid19.groupby("Country/Region")["Deaths"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

px.bar(
    top_countries,
    title="Bivariate: Top 10 Countries by Deaths"
).show()

In [ ]:
# Location vs Numeric

In [ ]:
px.scatter(
    covid19,
    x="Lat",
    y="Confirmed",
    title="Bivariate: Latitude vs Confirmed Cases"
).show()

px.scatter(
    covid19,
    x="Long",
    y="Confirmed",
    title="Bivariate: Longitude vs Confirmed Cases"
).show()

In [ ]:
# Multivariate Analysis

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_data["Date"],
    y=daily_data["Confirmed"],
    name="Confirmed"
))

fig.add_trace(go.Scatter(
    x=daily_data["Date"],
    y=daily_data["Recovered"],
    name="Recovered"
))

fig.add_trace(go.Scatter(
    x=daily_data["Date"],
    y=daily_data["Deaths"],
    name="Deaths"
))

fig.add_trace(go.Scatter(
    x=daily_data["Date"],
    y=daily_data["Active"],
    name="Active"
))

fig.update_layout(
    title="Multivariate Analysis: COVID-19 Impact Over Time",
    xaxis_title="Date",
    yaxis_title="Cases"
)

fig.show()

In [ ]:
# Prepare data for Prophet

In [ ]:
prophet_df = daily_data[["Date", "Confirmed"]]
prophet_df.columns = ["ds", "y"]

In [ ]:
# Train the model

In [ ]:
model = Prophet()
model.fit(prophet_df)

In [ ]:
# Predict next 7 days

In [ ]:
future = model.make_future_dataframe(periods=7)
forecast = model.predict(future)

In [ ]:
pred = forecast[['ds', 'yhat']]
actual_vs_pred = daily_data[['Date', 'Confirmed']].merge(
    forecast[['ds', 'yhat']],
    left_on='Date',
    right_on='ds',
    how='left'
)
actual_vs_pred.tail(15)

In [ ]:
# Visualize prediction

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=prophet_df["ds"],
    y=prophet_df["y"],
    name="Actual Cases"
))
fig.add_trace(go.Scatter(
    x=forecast["ds"],
    y=forecast["yhat"],
    name="Predicted Cases"
))
fig.update_layout(
    title="COVID-19 Prediction: Next 7 Days",
    xaxis_title="Date",
    yaxis_title="Confirmed Cases"
)
fig.show()